# Eval — Base Model (`unsloth/gemma-3-1b-it`)

**Run this on a fresh T4 Colab instance (separate from `eval_finetuned.ipynb`).**

This notebook:
1. Installs dependencies
2. Reconstructs the exact eval split (same seeds as training)
3. Runs the base model on 300 eval examples
4. Loads `finetuned_results.json` from `eval_finetuned.ipynb`
5. Prints the full side-by-side comparison

**Before running:** upload `finetuned_results.json` to this Colab session  
(Files panel on the left → Upload), or mount Drive if you saved it there.

In [ ]:
# ── Install ───────────────────────────────────────────────────────────────────
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch
    v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, '0.0.34')
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import json, re, sqlite3, time, torch
from tqdm import tqdm
from datasets import load_dataset
from unsloth import FastModel
from unsloth.chat_templates import get_chat_template

# ── Load populated mock database ─────────────────────────────
MOCK_DB_PATH = "/content/library_mock_data.sql"

with open(MOCK_DB_PATH, "r") as f:
    MOCK_DB_SQL = f.read()

print("Mock database loaded.")

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name} | VRAM: {props.total_memory / 1024**3:.1f} GB")

# Create Library SQL Evaluation Dataset (Hugging Face Format)

In [ ]:
import json

dataset = []

def add(q, sql, difficulty="easy", category="filter"):
    dataset.append({
        "question": q,
        "sql": sql,
        "difficulty": difficulty,
        "category": category
    })

# ─────────────────────────────────────────────
# EASY (SELECT / FILTER)
# ─────────────────────────────────────────────

add(
    "List all book titles in the library.",
    "SELECT title FROM books;",
    "easy", "select"
)

add(
    "Find all books written by J.R.R. Tolkien.",
    "SELECT title FROM books WHERE author = 'J.R.R. Tolkien';",
    "easy", "filter"
)

add(
    "Get all users in the system.",
    "SELECT name FROM users;",
    "easy", "select"
)

add(
    "Show all books published in 1954.",
    "SELECT title FROM books WHERE publication_year = 1954;",
    "easy", "filter"
)

add(
    "Find all reviews with rating equal to 5.",
    "SELECT * FROM reviews WHERE rating = 5;",
    "easy", "filter"
)

add(
    "Count total number of books.",
    "SELECT COUNT(*) FROM books;",
    "easy", "aggregation"
)

add(
    "Find the newest book in the library.",
    "SELECT title FROM books ORDER BY publication_year DESC LIMIT 1;",
    "easy", "order"
)

add(
    "Find the oldest book in the library.",
    "SELECT title FROM books ORDER BY publication_year ASC LIMIT 1;",
    "easy", "order"
)

add(
    "Get all checkout records.",
    "SELECT * FROM checkout;",
    "easy", "select"
)

add(
    "Find all users who joined after 2023-06-01.",
    "SELECT name FROM users WHERE join_date > '2023-06-01';",
    "easy", "filter"
)

# ─────────────────────────────────────────────
# MEDIUM (JOINS / GROUP BY)
# ─────────────────────────────────────────────

add(
    "List all books reviewed by user 1.",
    "SELECT b.title FROM books b JOIN reviews r ON b.id = r.book_id WHERE r.user_id = 1;",
    "medium", "join"
)

add(
    "Find all users who have written reviews.",
    "SELECT DISTINCT u.name FROM users u JOIN reviews r ON u.user_id = r.user_id;",
    "medium", "join"
)

add(
    "Get average rating per book.",
    "SELECT book_id, AVG(rating) FROM reviews GROUP BY book_id;",
    "medium", "aggregation"
)

add(
    "Find books with at least 3 reviews.",
    "SELECT book_id FROM reviews GROUP BY book_id HAVING COUNT(*) >= 3;",
    "medium", "aggregation"
)

add(
    "Find users who gave ratings less than 3.",
    "SELECT DISTINCT user_id FROM reviews WHERE rating < 3;",
    "medium", "filter"
)

add(
    "Find books that have never been checked out.",
    "SELECT title FROM books WHERE title NOT IN (SELECT title FROM checkout);",
    "medium", "subquery"
)

add(
    "Find most reviewed book.",
    "SELECT book_id FROM reviews GROUP BY book_id ORDER BY COUNT(*) DESC LIMIT 1;",
    "medium", "aggregation"
)

add(
    "Get all checkout records for March 2026.",
    "SELECT * FROM checkout WHERE checkout_date LIKE '2026-03%';",
    "medium", "filter"
)

add(
    "Find number of reviews per user.",
    "SELECT user_id, COUNT(*) FROM reviews GROUP BY user_id;",
    "medium", "aggregation"
)

add(
    "Find books reviewed by user named Alice Johnson.",
    "SELECT b.title FROM books b JOIN reviews r ON b.id = r.book_id JOIN users u ON u.user_id = r.user_id WHERE u.name = 'Alice Johnson';",
    "medium", "join"
)

# ─────────────────────────────────────────────
# HARD (MULTI-JOIN / COMPLEX LOGIC)
# ─────────────────────────────────────────────

add(
    "Find users who reviewed The Hobbit.",
    "SELECT u.name FROM users u JOIN reviews r ON u.user_id = r.user_id JOIN books b ON b.id = r.book_id WHERE b.title = 'The Hobbit';",
    "hard", "join"
)

add(
    "Get top 3 highest rated books.",
    "SELECT book_id, AVG(rating) as avg_rating FROM reviews GROUP BY book_id ORDER BY avg_rating DESC LIMIT 3;",
    "hard", "aggregation"
)

add(
    "Find books with no reviews.",
    "SELECT b.title FROM books b LEFT JOIN reviews r ON b.id = r.book_id WHERE r.review_id IS NULL;",
    "hard", "join"
)

add(
    "Find users who reviewed more than 2 books.",
    "SELECT user_id FROM reviews GROUP BY user_id HAVING COUNT(DISTINCT book_id) > 2;",
    "hard", "aggregation"
)

add(
    "Find average rating given by each user.",
    "SELECT user_id, AVG(rating) FROM reviews GROUP BY user_id;",
    "hard", "aggregation"
)

# ─────────────────────────────────────────────
# SAVE JSON
# ─────────────────────────────────────────────

output_path = "/content/library_benchmark.json"

with open(output_path, "w") as f:
    json.dump(dataset, f, indent=2)

print(f"Saved {len(dataset)} examples to {output_path}")

## 1. Load Fine-tuned Results
Load the JSON file saved by `eval_finetuned.ipynb` before running the rest.

In [ ]:
# ── Try Drive first, fall back to local upload ────────────────────────────────
ft_results = None

drive_path = "/content/drive/MyDrive/finetuned_results.json"
local_path = "/content/finetuned_results.json"

for path in [local_path, drive_path]:
    try:
        with open(path) as f:
            ft_results = json.load(f)
        print(f"Loaded fine-tuned results from: {path}")
        break
    except FileNotFoundError:
        continue

if ft_results is None:
    print("WARNING: finetuned_results.json not found.")
    print("Upload it via Files panel or mount Drive, then re-run this cell.")
    print("The eval will still run — comparison table will show N/A for fine-tuned.")
else:
    print(f"Fine-tuned model: {ft_results['label']}")
    print(f"  Exact Match:        {ft_results['exact_match']:.1%}")
    print(f"  Execution Accuracy: {ft_results['exec_accuracy']:.1%}")
    print(f"  Syntax Error Rate:  {ft_results['syntax_error_rate']:.1%}")

## 2. Reconstruct Eval Split

In [ ]:
# ── Load base model (we need the tokenizer to replicate the filter) ───────────
model, tokenizer = FastModel.from_pretrained(
    model_name     = "unsloth/gemma-3-1b-it-unsloth-bnb-4bit",
    max_seq_length = 512,
    load_in_4bit   = True,
)
tokenizer = get_chat_template(tokenizer, chat_template="gemma-3")
FastModel.for_inference(model)
print("Base model loaded.")

In [ ]:
# ── Replicate training data
from datasets import Dataset, DatasetDict
import json

# ── Load YOUR benchmark dataset
with open("/content/library_benchmark.json", "r") as f:
    data = json.load(f)

dataset = Dataset.from_list(data)

# ── Train / Test split
dataset = dataset.train_test_split(test_size=0.2, seed=42)

dataset = DatasetDict({
    "train": dataset["train"],
    "test": dataset["test"]
})

eval_dataset = dataset["test"]

print(dataset)
print(f"Eval split size: {len(eval_dataset)} examples")
print(f"Eval split size: {len(eval_dataset)} examples")

## 3. Eval Helpers

In [ ]:
def normalize_sql(sql: str) -> str:
    sql = sql.strip().rstrip(";")
    sql = re.sub(r"```(?:sql)?\s*", "", sql, flags=re.IGNORECASE)
    sql = re.sub(r"```", "", sql)
    sql = sql.strip().lower()
    sql = re.sub(r"\s+", " ", sql)
    sql = re.sub(r"\binner join\b", "join", sql)
    return sql


def extract_sql(raw_output: str) -> str:
    """Pull SQL even if model wrapped it in explanation text."""
    fence = re.search(r"```(?:sql)?\s*([\s\S]+?)```", raw_output, re.IGNORECASE)
    if fence:
        return fence.group(1).strip()
    for line in raw_output.splitlines():
        line = line.strip()
        if re.match(r"^(SELECT|INSERT|UPDATE|DELETE|WITH|CREATE)", line, re.IGNORECASE):
            return line
    return raw_output.strip()


def execute_sql(query: str):
    try:
        conn = sqlite3.connect(":memory:")

        # Build full populated database
        conn.executescript(MOCK_DB_SQL)

        rows = conn.execute(query).fetchall()

        conn.close()

        return rows

    except Exception:
        return None


SYSTEM_PROMPT = (
    "Generate one SQL query that answers the question "
    "using only the schema below.\n"
    "Return only SQL. Do not include markdown, comments, or explanation."
)

def generate_sql(model, tokenizer, schema: str, question: str) -> str:
    messages = [{
        "role": "user",
        "content": [{
            "type": "text",
            "text": f"{SYSTEM_PROMPT}\n\nSchema:\n{schema}\n\nQuestion: {question}"
        }]
    }]
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_tensors="pt",
        return_dict=True,
    ).to("cuda")

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=128,
            temperature=0.0,
            do_sample=False,
        )

    new_tokens = out[0][inputs["input_ids"].shape[1]:]
    raw = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    return extract_sql(raw)


print("Helpers defined.")

## 4. Run Base Model Evaluation

In [ ]:
# ── Run eval on base model ────────────────────────────────────────────────────
N_EVAL = 300

samples = eval_dataset.select(range(min(N_EVAL, len(eval_dataset))))

exact_matches = 0
exec_matches  = 0
syntax_errors = 0
wrong_results = 0
failures      = []

t0 = time.time()

for i, row in enumerate(tqdm(samples, desc="base model")):

    # Keep schema ONLY for prompting the model
    question = row["question"]
    gold     = row["sql"]

    # Generate SQL prediction
    SYSTEM_PROMPT = """
    You are a SQL generator.

    Database schema:
    books(id, title, author, publication_year)
    users(user_id, name, dob, join_date)
    reviews(review_id, user_id, book_id, rating, review_text, review_date)
    checkout(checkout_id, title, user_id, checkout_date, checkin_date)

    Return only SQL.
    """

    pred = generate_sql(model, tokenizer, SYSTEM_PROMPT, question)

    # ── Exact Match
    em = normalize_sql(pred) == normalize_sql(gold)

    if em:
        exact_matches += 1

    # ── Execution Accuracy
    # execute_sql() now ONLY takes the query
    # because the populated DB is loaded internally

    gold_result = execute_sql(gold)
    pred_result = execute_sql(pred)

    # ── Syntax Errors
    if pred_result is None:

        syntax_errors += 1

        failures.append({
            "idx": i,
            "error": "syntax_error",
            "question": question,
            "schema": schema,
            "gold": gold,
            "pred": pred
        })

    # ── Correct Execution
    elif pred_result == gold_result:

        exec_matches += 1

    # ── Wrong Semantic Result
    else:

        wrong_results += 1

        failures.append({
            "idx": i,
            "error": "wrong_result",
            "question": question,
            "schema": schema,
            "gold": gold,
            "pred": pred,
            "gold_result": gold_result,
            "pred_result": pred_result
        })

# ── Metrics
total   = len(samples)
elapsed = time.time() - t0

base_results = {
    "label":             "unsloth/gemma-3-1b-it (base)",
    "n":                 total,
    "exact_match":       exact_matches / total,
    "exec_accuracy":     exec_matches / total,
    "syntax_error_rate": syntax_errors / total,
    "wrong_result_rate": wrong_results / total,
    "elapsed_s":         round(elapsed, 1),
    "failures":          failures,
}

# ── Print Results
print(f"\n{'='*60}")
print(f"  Base model evaluation (n={total})")
print(f"{'='*60}")

print(
    f"  Exact Match:        "
    f"{exact_matches:3d}/{total} = "
    f"{base_results['exact_match']:.1%}"
)

print(
    f"  Execution Accuracy: "
    f"{exec_matches:3d}/{total} = "
    f"{base_results['exec_accuracy']:.1%}"
)

print(
    f"  Syntax Errors:      "
    f"{syntax_errors:3d}/{total} = "
    f"{base_results['syntax_error_rate']:.1%}"
)

print(
    f"  Wrong Results:      "
    f"{wrong_results:3d}/{total} = "
    f"{base_results['wrong_result_rate']:.1%}"
)

print(
    f"  Eval time:          "
    f"{elapsed:.0f}s "
    f"({elapsed/total:.2f}s/sample)"
)

# ── Failure Breakdown
syntax_fails = [
    f for f in failures
    if f["error"] == "syntax_error"
]

wrong_fails = [
    f for f in failures
    if f["error"] == "wrong_result"
]

print(f"\nTotal failures: {len(failures)}/{total}")
print(f"  Syntax errors: {len(syntax_fails)}")
print(f"  Wrong results: {len(wrong_fails)}")

# ── Sample Syntax Errors
print("\n── Sample syntax errors (first 3) ──")

for f in syntax_fails[:3]:

    print(f"\nQ: {f['question']}")
    print(f"Gold SQL: {f['gold']}")
    print(f"Pred SQL: {f['pred']}")

# ── Sample Wrong Results
print("\n── Sample wrong-result queries (first 3) ──")

for f in wrong_fails[:3]:

    print(f"\nQ: {f['question']}")
    print(f"Gold SQL: {f['gold']}")
    print(f"Pred SQL: {f['pred']}")

    print(f"\nGold Result:")
    print(f"{f['gold_result']}")

    print(f"\nPred Result:")
    print(f"{f['pred_result']}")

## 5. Base Model Failure Analysis

In [ ]:
syntax_fails = [f for f in failures if f["error"] == "syntax_error"]
wrong_fails  = [f for f in failures if f["error"] == "wrong_result"]

print(f"Total failures: {len(failures)}/{total}")
print(f"  Syntax errors: {len(syntax_fails)}")
print(f"  Wrong results: {len(wrong_fails)}")

print("\n── Sample syntax errors (first 3) ──")
for f in syntax_fails[:3]:
    print(f"\nQ: {f['question']}")
    print(f"Gold: {f['gold']}")
    print(f"Pred: {f['pred']}")

print("\n── Sample wrong results (first 3) ──")
for f in wrong_fails[:3]:
    print(f"\nQ: {f['question']}")
    print(f"Gold: {f['gold']}")
    print(f"Pred: {f['pred']}")

In [ ]:
# ── Keyword-level failure breakdown ──────────────────────────────────────────
keywords = ["JOIN", "GROUP BY", "HAVING", "ORDER BY", "COUNT", "SUM", "AVG", "MAX", "MIN", "DISTINCT"]

print("── Failure rate by SQL keyword (base model) ──")
print(f"{'Keyword':<12} {'Fails':>6} {'Total':>6} {'Fail%':>7}")
print("-" * 35)

for kw in keywords:
    kw_samples = [r for r in samples if kw.lower() in r["answer"].lower()]
    kw_fails   = [f for f in failures if kw.lower() in f["gold"].lower()]
    if kw_samples:
        rate = len(kw_fails) / len(kw_samples)
        print(f"{kw:<12} {len(kw_fails):>6} {len(kw_samples):>6} {rate:>7.1%}")

## 6. Side-by-Side Comparison

In [ ]:
# ── Final comparison table ────────────────────────────────────────────────────
if ft_results is None:
    print("Fine-tuned results not loaded — run eval_finetuned.ipynb first.")
else:
    metrics = [
        ("Exact Match",       "exact_match",       True),
        ("Execution Accuracy","exec_accuracy",      True),
        ("Syntax Error Rate", "syntax_error_rate",  False),  # lower is better
        ("Wrong Result Rate", "wrong_result_rate",  False),
    ]

    w = 56
    print("\n" + "=" * w)
    print(f"  {'METRIC':<22} {'FINE-TUNED':>12} {'BASE':>10} {'DELTA':>8}")
    print("=" * w)

    for label, key, higher_better in metrics:
        ft_val   = ft_results[key]
        base_val = base_results[key]
        delta    = ft_val - base_val
        sign     = "+" if delta > 0 else ""
        # mark direction with arrow
        good = (delta > 0) == higher_better
        arrow = "▲" if good else "▼"
        print(f"  {label:<22} {ft_val:>11.1%} {base_val:>9.1%} {sign}{delta:>6.1%} {arrow}")

    print("=" * w)
    print(f"  Fine-tuned n={ft_results['n']} | Base n={base_results['n']}")
    print(f"  Fine-tuned eval time: {ft_results['elapsed_s']:.0f}s")
    print(f"  Base eval time:       {base_results['elapsed_s']:.0f}s")
    print("=" * w)

    # ── Verdict ───────────────────────────────────────────────────────────────
    exec_lift = ft_results["exec_accuracy"] - base_results["exec_accuracy"]
    print(f"\nExecution accuracy lift: {exec_lift:+.1%}")

    if exec_lift >= 0.35:
        print("Verdict: strong improvement — fine-tuning worked well.")
    elif exec_lift >= 0.20:
        print("Verdict: solid improvement — consider a second epoch for more gains.")
    elif exec_lift >= 0.10:
        print("Verdict: modest improvement — check syntax error rate and prompt alignment.")
    else:
        print("Verdict: low lift — investigate prompt format mismatch between train and eval.")

In [ ]:
# ── Save combined results ─────────────────────────────────────────────────────
combined = {
    "finetuned": {k: v for k, v in ft_results.items() if k != "failures"} if ft_results else None,
    "base":      {k: v for k, v in base_results.items() if k != "failures"},
}

with open("eval_comparison.json", "w") as f:
    json.dump(combined, f, indent=2)

print("Combined results saved to eval_comparison.json")

try:
    import shutil
    shutil.copy("eval_comparison.json", "/content/drive/MyDrive/eval_comparison.json")
    print("Also copied to Google Drive.")
except Exception:
    print("(Drive not mounted — download manually from Files panel)")

# ── GPU memory summary ────────────────────────────────────────────────────────
used_gb  = torch.cuda.max_memory_reserved() / 1024**3
total_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"\nPeak VRAM used: {used_gb:.2f} GB / {total_gb:.1f} GB ({used_gb/total_gb:.1%})")